This documentation is exclusively focused on using **MLflow** and **DagsHub** to track machine learning experiments, including parameters, metrics, and model artifacts.

# MLflow + DagsHub: Experiment Tracking & Artifact Management

To build a professional MLOps pipeline, you must have a central server to record your training runs. DagsHub provides a managed MLflow server for every repository, allowing you to log everything from hyperparameters to serialized model files (`.pkl`, `.joblib`) without setting up your own infrastructure.

---

### Step 1: Connect Your Repository to DagsHub

DagsHub serves as the central hub where your code and MLflow experiments converge.

1. **Sign up**: Create an account at [DagsHub.com](https://dagshub.com).
2. **Import Repository**: Click **+ Create** → **Import Repository**.
3. **Mirror GitHub**: Select your `NGForexCast` repository and ensure **Mirror** mode is active. This links your experiments directly to your code commits.

---

### Step 2: Instrument Your Code for Tracking

You must explicitly point the MLflow client to DagsHub's remote server and authenticate using your access token.

**File: `src/model/train.py**`

```python
import mlflow
import dagshub
import os

def run_train():
    # 1. Explicitly set the tracking URI
    # Pattern: https://dagshub.com/<owner>/<repo>.mlflow
    repo_owner = "Chiebukar"
    repo_name = "NGForexCast"
    mlflow.set_tracking_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
    
    # 2. Organize runs under a specific experiment
    mlflow.set_experiment("Forex_Model_Retraining")

    # 3. Headless Authentication
    # This uses the token provided by your orchestration environment
    token = os.environ.get("DAGSHUB_TOKEN")
    dagshub.auth.add_app_token(token)
    
    with mlflow.start_run(run_name="retrain_cycle"):
        # Log Parameters (Hyperparameters)
        mlflow.log_param("model_flavor", "LightGBM")
        mlflow.log_param("n_estimators", 100)
        
        # Log Metrics (Performance)
        mlflow.log_metric("rmse", 0.045)
        
        # 4. Log Artifacts (The Model File)
        # It is best practice to log the actual model file for versioning
        mlflow.sklearn.log_model(model, "forex_model_v1")
        
        # Optional: Log extra files (plots, CSVs, logs)
        # mlflow.log_artifact("plots/confusion_matrix.png")

```

---

### Step 3: Providing Credentials during Orchestration

When you automate this training run using **Prefect**, you must inject the `DAGSHUB_TOKEN` as an environment variable. This allows the remote worker to authenticate with DagsHub headlessly.

**Deployment Command:**

```bash
uvx prefect-cloud deploy src/orchestration/flows.py:autonomous_forex_ai \
 --name ngn_forex_autonomous \
 --with-requirements requirements.txt \
 --env DAGSHUB_TOKEN="your_dagshub_token_here" \
 --env EXCHANGE_RATE_API="..." \
 --env SUPABASE_DB_URL="..."

```

---

### Why this is Robust for Artifacts

* **Model Lineage**: DagsHub automatically links the model artifact in MLflow to the GitHub commit that produced it.
* **Artifact Store**: You don't need an S3 bucket or Azure Blob storage; DagsHub acts as the artifact store, hosting your `.pkl` and `.joblib` files for free.
* **Collaboration**: Team members can download your best-performing models directly from the DagsHub UI or via the MLflow API for deployment.

[DagsHub MLflow Experiment Tracking Tutorial](https://www.youtube.com/watch?v=6ngxBkx05Fs)

This video provides a practical walkthrough on setting up DagsHub to act as your remote MLflow tracking server, specifically showing how parameters and artifacts appear in the UI.